In [ ]:
# ── Notebook environment setup ───────────────────────────────────────────
import sys, os
from pathlib import Path

# Locate project root: look for config.py in cwd or parent (handles both
# running from notebooks/ and running from project root)
_cwd = Path(os.getcwd())
ROOT = _cwd if (_cwd / "config.py").exists() else _cwd.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import (
    ROOT_DIR, BOOK_DEPTH_DIR, TRADES_DIR, PROCESSED_DIR,
    ALL_PAIRS_CLEANED, ALL_PAIRS_TRADES, ALL_PAIRS_LABELED,
    DEFAULT_PAIRS, DEFAULT_PERIOD,
    ROLL_WINDOW_SHORT, ROLL_WINDOW_REGIME,
    CRASH_HORIZON, CRASH_SIGMA, LABEL_PAIR, SEQ_LEN,
)
print(f"Project root: {ROOT}")
print(f"Data dir:     {BOOK_DEPTH_DIR}")

In [ ]:
from models import SequenceDataset
ds = SequenceDataset(str(ALL_PAIRS_LABELED), seq_len=SEQ_LEN, label_pair=LABEL_PAIR)
ds.load()

# Show class balance
import pandas as pd
import numpy as np
df_info = pd.read_csv(str(ALL_PAIRS_LABELED), nrows=5)
label_col = f"{LABEL_PAIR}_flash_crash_label"
df_all = pd.read_csv(str(ALL_PAIRS_LABELED))
counts = df_all[label_col].value_counts().sort_index()
print("Class balance:")
for cls, cnt in counts.items():
    print(f"  {cls}: {cnt} ({100*cnt/len(df_all):.2f}%)")

In [ ]:
(X_tr, y_tr), (X_cal, y_cal), (X_te, y_te) = ds.get_splits()
print(f"Train:  X={X_tr.shape}, y={y_tr.shape}")
print(f"Cal:    X={X_cal.shape}, y={y_cal.shape}")
print(f"Test:   X={X_te.shape}, y={y_te.shape}")

In [ ]:
from models import STanHopModel, HopCPT

ALPHA = 0.1  # miscoverage level

model = STanHopModel(seq_len=SEQ_LEN, n_features=ds.n_features)
cpt = HopCPT(model, alpha=ALPHA)
cpt.fit(X_tr, y_tr, X_val=X_cal, y_val=y_cal)
cpt.calibrate(X_cal, y_cal)

In [ ]:
metrics = cpt.evaluate(X_te, y_te)
for k, v in metrics.items():
    print(f"  {k:25s}: {v:.4f}")

sets = cpt.predict_set(X_te)
import numpy as np
print(f"Prediction set breakdown:")
print(f"  Crash only (1):     {(sets==1).sum()}")
print(f"  No crash only (0):  {(sets==0).sum()}")
print(f"  Uncertain (2):      {(sets==2).sum()}")
print(f"  Empty set (-1):     {(sets==-1).sum()}")

In [ ]:
from models import MLBaselinesModel
(X_tr_f, y_tr_f), (X_cal_f, y_cal_f), (X_te_f, y_te_f) = ds.get_flat_splits()
xgb = MLBaselinesModel("xgboost")
xgb.fit(X_tr_f, y_tr_f)
xgb_metrics = xgb.evaluate(X_te_f, y_te_f)
print("XGBoost baseline:")
for k, v in xgb_metrics.items():
    print(f"  {k:25s}: {v:.4f}")